In [9]:
import os
import csv
import time
from pathlib import Path
from typing import List, Tuple

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import transforms
from PIL import Image

DATASET_DIR = Path("dataset")
IMG_DIR = DATASET_DIR / "images"
CSV_PATH = DATASET_DIR / "labels.csv"

IMG_W, IMG_H = 168, 32
BATCH_SIZE = 128
EPOCHS = 15
LR = 3e-4
NUM_WORKERS = 4
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

ALPHABET = "ABCDEFGHIJKLMNOPQRSTUVWXYZ"
char2idx = {c: i + 1 for i, c in enumerate(ALPHABET)}
idx2char = {i + 1: c for i, c in enumerate(ALPHABET)}
BLANK_IDX = 0
NUM_CLASSES = 1 + len(ALPHABET)

In [6]:
def encode_text(text: str) -> torch.Tensor:
    return torch.tensor([char2idx[c] for c in text], dtype=torch.long)

@torch.no_grad()
def ctc_greedy_decode(logits: torch.Tensor) -> List[str]:
    preds = logits.argmax(dim=2)  # [T, B]
    T, B = preds.shape
    out = []
    for b in range(B):
        seq = preds[:, b].tolist()
        dedup = []
        prev = None
        for s in seq:
            if s != prev:
                dedup.append(s)
            prev = s
        chars = [idx2char[i] for i in dedup if i != BLANK_IDX]
        out.append("".join(chars))
    return out

def edit_distance(a: str, b: str) -> int:
    n, m = len(a), len(b)
    dp = list(range(m + 1))
    for i in range(1, n + 1):
        prev = dp[0]
        dp[0] = i
        for j in range(1, m + 1):
            cur = dp[j]
            cost = 0 if a[i - 1] == b[j - 1] else 1
            dp[j] = min(dp[j] + 1, dp[j - 1] + 1, prev + cost)
            prev = cur
    return dp[m]

@torch.no_grad()
def cer(pred: str, gt: str) -> float:
    if len(gt) == 0:
        return 0.0 if len(pred) == 0 else 1.0
    return edit_distance(pred, gt) / len(gt)

class OCRDataset(Dataset):
    def __init__(self, img_dir: Path, csv_path: Path):
        self.img_dir = img_dir
        self.samples: List[Tuple[str, str]] = []

        with open(csv_path, "r", encoding="utf-8") as f:
            reader = csv.DictReader(f)
            for row in reader:
                fn = row["filename"]
                text = row["text"].strip()
                if 4 <= len(text) <= 8:
                    self.samples.append((fn, text))

        self.tf = transforms.Compose([
            transforms.Grayscale(num_output_channels=1),
            transforms.Resize((IMG_H, IMG_W)),  # H,W
            transforms.ToTensor(),              # [0,1]
            transforms.Normalize(mean=[0.5], std=[0.5]),  # [-1,1]
        ])

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx: int):
        fn, text = self.samples[idx]
        img_path = self.img_dir / fn
        img = Image.open(img_path).convert("L")
        img = self.tf(img)  # [1,H,W]
        target = encode_text(text)
        return img, target, text


def collate_fn(batch):
    """
    batch: list of (img, target_tensor, text)
    vraća:
      images: [B,1,H,W]
      targets: 1D concatenated
      target_lengths: [B]
      texts: list[str]
    """
    imgs, targets, texts = zip(*batch)
    imgs = torch.stack(imgs, dim=0)

    target_lengths = torch.tensor([t.numel() for t in targets], dtype=torch.long)
    targets_concat = torch.cat(targets, dim=0)
    return imgs, targets_concat, target_lengths, list(texts)

In [7]:
class CRNN(nn.Module):
    def __init__(self, num_classes: int):
        super().__init__()

        # CNN koji smanjuje visinu na 1
        self.cnn = nn.Sequential(
            # [B,1,32,168]
            nn.Conv2d(1, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(),
            nn.MaxPool2d(2, 2),  # [B,64,16,84]

            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(),
            nn.MaxPool2d(2, 2),  # [B,128,8,42]

            nn.Conv2d(128, 256, 3, padding=1), nn.BatchNorm2d(256), nn.ReLU(),
            nn.Conv2d(256, 256, 3, padding=1), nn.BatchNorm2d(256), nn.ReLU(),
            nn.MaxPool2d((2, 1), (2, 1)),  # [B,256,4,42]

            nn.Conv2d(256, 512, 3, padding=1), nn.BatchNorm2d(512), nn.ReLU(),
            nn.Conv2d(512, 512, 3, padding=1), nn.BatchNorm2d(512), nn.ReLU(),
            nn.MaxPool2d((2, 1), (2, 1)),  # [B,512,2,42]

            nn.Conv2d(512, 512, 2, padding=0), nn.BatchNorm2d(512), nn.ReLU(),
            # kernel 2x2 -> [B,512,1,41]  (širina se malo smanji)
        )

        # RNN preko širine (time steps)
        self.rnn = nn.LSTM(
            input_size=512,
            hidden_size=256,
            num_layers=2,
            bidirectional=True,
            batch_first=False
        )

        self.fc = nn.Linear(256 * 2, num_classes)

    def forward(self, x):
        # x: [B,1,H,W]
        f = self.cnn(x)            # [B,512,1,T]
        f = f.squeeze(2)           # [B,512,T]
        f = f.permute(2, 0, 1)     # [T,B,512]

        y, _ = self.rnn(f)         # [T,B,512]
        logits = self.fc(y)        # [T,B,C]
        return logits

def train_one_epoch(model, loader, optimizer, scaler, ctc_loss):
    model.train()
    total_loss = 0.0

    for imgs, targets, target_lengths, _texts in loader:
        imgs = imgs.to(DEVICE, non_blocking=True)
        targets = targets.to(DEVICE, non_blocking=True)
        target_lengths = target_lengths.to(DEVICE, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with torch.amp.autocast("cuda", enabled=(DEVICE == "cuda")):
            logits = model(imgs)               # [T,B,C]
            log_probs = F.log_softmax(logits, dim=2)

            T, B, _ = log_probs.shape
            input_lengths = torch.full(size=(B,), fill_value=T, dtype=torch.long, device=DEVICE)

            loss = ctc_loss(log_probs, targets, input_lengths, target_lengths)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item()

    return total_loss / max(1, len(loader))


@torch.no_grad()
def evaluate(model, loader, ctc_loss, max_batches=50):
    model.eval()
    total_loss = 0.0
    total_cer = 0.0
    n = 0

    for bi, (imgs, targets, target_lengths, texts) in enumerate(loader):
        imgs = imgs.to(DEVICE, non_blocking=True)
        targets = targets.to(DEVICE, non_blocking=True)
        target_lengths = target_lengths.to(DEVICE, non_blocking=True)

        logits = model(imgs)
        log_probs = F.log_softmax(logits, dim=2)
        T, B, _ = log_probs.shape
        input_lengths = torch.full((B,), T, dtype=torch.long, device=DEVICE)

        loss = ctc_loss(log_probs, targets, input_lengths, target_lengths)
        total_loss += loss.item()

        preds = ctc_greedy_decode(logits)
        for p, gt in zip(preds, texts):
            total_cer += cer(p, gt)
            n += 1

        if bi + 1 >= max_batches:
            break

    return total_loss / max(1, min(len(loader), max_batches)), (total_cer / max(1, n))

In [10]:
ds = OCRDataset(IMG_DIR, CSV_PATH)
start_time = time.time()

val_size = int(0.1 * len(ds))
train_size = len(ds) - val_size
train_ds, val_ds = random_split(ds, [train_size, val_size], generator=torch.Generator().manual_seed(42))

train_loader = DataLoader(
    train_ds, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=NUM_WORKERS, pin_memory=True, collate_fn=collate_fn)

val_loader = DataLoader(
    val_ds, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=True, collate_fn=collate_fn)

model = CRNN(NUM_CLASSES).to(DEVICE)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=EPOCHS,
    eta_min=LR * 0.05
)
scaler = torch.amp.GradScaler("cuda", enabled=(DEVICE == "cuda"))
ctc_loss = nn.CTCLoss(blank=BLANK_IDX, zero_infinity=True)

best_val_cer = 1e9
os.makedirs("checkpoints", exist_ok=True)

for epoch in range(1, EPOCHS + 1):
    tr_loss = train_one_epoch(model, train_loader, optimizer, scaler, ctc_loss)
    va_loss, va_cer = evaluate(model, val_loader, ctc_loss)

    scheduler.step()

    print(f"Epoch {epoch:02d} | train_loss={tr_loss:.4f} | val_loss={va_loss:.4f} | val_CER={va_cer:.4f} | lr={scheduler.get_last_lr()[0]:.6f}")

    if va_cer < best_val_cer:
        best_val_cer = va_cer
        torch.save({
            "model": model.state_dict(),
            "alphabet": ALPHABET,
            "img_w": IMG_W,
            "img_h": IMG_H
        }, "checkpoints/crnn_ctc_best.pt")
        print("saved checkpoints/crnn_ctc_best.pt")

print("Best val CER:", best_val_cer)
total_time = time.time() - start_time
print(f"\nTotal training time: {total_time:.2f} seconds")

/tmp/ipython-input-3328465516.py:58: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=(DEVICE == "cuda")):


Epoch 01 | train_loss=2.6903 | val_loss=0.0294 | val_CER=0.0001 | lr=0.000297
saved checkpoints/crnn_ctc_best.pt
Epoch 02 | train_loss=0.0108 | val_loss=0.0051 | val_CER=0.0001 | lr=0.000288
saved checkpoints/crnn_ctc_best.pt
Epoch 03 | train_loss=0.0383 | val_loss=0.0066 | val_CER=0.0002 | lr=0.000273
Epoch 04 | train_loss=0.0035 | val_loss=0.0022 | val_CER=0.0001 | lr=0.000253
Epoch 05 | train_loss=0.0016 | val_loss=0.0014 | val_CER=0.0001 | lr=0.000229
saved checkpoints/crnn_ctc_best.pt
Epoch 06 | train_loss=0.0011 | val_loss=0.0011 | val_CER=0.0001 | lr=0.000202
Epoch 07 | train_loss=0.0008 | val_loss=0.0010 | val_CER=0.0001 | lr=0.000172
Epoch 08 | train_loss=0.0006 | val_loss=0.0008 | val_CER=0.0001 | lr=0.000143
Epoch 09 | train_loss=0.0005 | val_loss=0.0007 | val_CER=0.0001 | lr=0.000113
Epoch 10 | train_loss=0.0004 | val_loss=0.0006 | val_CER=0.0001 | lr=0.000086
Epoch 11 | train_loss=0.0004 | val_loss=0.0006 | val_CER=0.0001 | lr=0.000062
Epoch 12 | train_loss=0.0003 | val_lo